
# Module 2 Assignment — Data Cleaning and Reshaping

**Course:** Data Science and Cloud Computing for Bioinformatics  
**Dataset:** `bioinformatics_messy_data_module_2.csv.xlsx`

This notebook follows Tasks 1–9 of the assignment. The dataset is deliberately messy, so problems are identified before correction. The final cleaned dataset retains the GeneA value of 5000 as a flagged biological/technical outlier rather than deleting it automatically.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

file_path = "bioinformatics_messy_data_module_2.csv.xlsx"
df = pd.read_excel(file_path)

print("Dataset loaded successfully.")


## Task 1 — Load and Inspect the Dataset

Each row represents a biological sample. Columns contain sample metadata and expression values for GeneA, GeneB and GeneC.

In [ ]:

df.head()
df.tail()
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.info()
print(df.describe(include="all"))
print(df.dtypes)


In [ ]:

# Initial problem table
problems = pd.DataFrame({
    "Problem": [
        "Missing values",
        "Duplicate record/sample identifier",
        "Inconsistent Species capitalization",
        "Inconsistent Tissue capitalization",
        "Inconsistent Sex labels",
        "Inconsistent Treatment labels",
        "Invalid biological age",
        "Extreme GeneA value"
    ],
    "Variable": ["Age, GeneB, GeneC","Sample_ID","Species","Tissue","Sex","Treatment","Age","GeneA"],
    "Example": ["Age=S005 missing; GeneB=S003 missing; GeneC=S010/S010 missing",
                "S010 occurs twice",
                "Homo sapiens / homo sapiens / Homo Sapiens",
                "Liver / liver / LIVER",
                "M / Male / F / Female / male / FEMALE",
                "Control / control / CTRL / Treated / treated / TRT",
                "S013 Age=-5",
                "S009 GeneA=5000"],
    "Proposed solution": [
        "Investigate and impute where scientifically defensible",
        "Remove the exact duplicate after verification",
        "Standardize to Homo sapiens",
        "Standardize case to Liver/Heart/Kidney",
        "Standardize to Male/Female",
        "Standardize to Control/Treated",
        "Treat as invalid/missing; do not invent an exact age",
        "Flag as potential outlier; investigate rather than automatically delete"
    ]
})
problems


## Task 2 — Missing Data

Missingness is quantified first. For this small dataset, median imputation is used for numerical variables because it is robust to the extreme GeneA value and avoids deleting samples.

In [ ]:

print("Missing counts:")
print(df.isnull().sum())

missing_pct = (df.isnull().mean() * 100).round(2)
print("\nMissing percentages:")
print(missing_pct)


In [ ]:

# Missing-value treatment is performed after duplicate removal.
# Median values are calculated from the observed values only.
df_clean = df.drop_duplicates().copy()

print("Median Age:", df_clean["Age"].median())
print("Median GeneB:", df_clean["GeneB"].median())
print("Median GeneC:", df_clean["GeneC"].median())


## Task 3 — Detect and Handle Duplicates

S010 is duplicated and the two rows are exact duplicates, so one copy is removed. In a real RNA-seq experiment, repeated samples may instead be legitimate technical replicates and should not be removed without checking sample metadata and experimental design.

In [ ]:

print("Duplicate rows:", df.duplicated().sum())
print("Duplicated Sample_ID values:")
print(df.loc[df["Sample_ID"].duplicated(keep=False), ["Sample_ID"]])

df_clean = df.drop_duplicates().copy()
print("Rows after removing the exact duplicate:", len(df_clean))


## Task 4 — Standardize Categorical Variables

In [ ]:

for col in ["Species", "Tissue", "Sex", "Treatment"]:
    print(f"\n{col} before cleaning:")
    print(df_clean[col].value_counts())
    print("Unique:", df_clean[col].unique())


In [ ]:

df_clean["Species"] = df_clean["Species"].astype(str).str.strip().str.lower().replace(
    {"homo sapiens": "Homo sapiens"}
)
df_clean["Tissue"] = df_clean["Tissue"].astype(str).str.strip().str.lower().str.capitalize()
df_clean["Sex"] = (
    df_clean["Sex"].astype(str).str.strip().str.lower()
    .replace({"m": "Male", "male": "Male", "f": "Female", "female": "Female"})
)
df_clean["Treatment"] = (
    df_clean["Treatment"].astype(str).str.strip().str.lower()
    .replace({"control": "Control", "ctrl": "Control",
              "treated": "Treated", "trt": "Treated"})
)

for col in ["Species", "Tissue", "Sex", "Treatment"]:
    print(f"\n{col} after cleaning:")
    print(df_clean[col].value_counts())


## Task 5 — Identify Invalid Biological Values

S013 has Age = -5. A negative age is biologically impossible. The original value cannot be scientifically corrected from this dataset alone. It is therefore converted to missing and then treated under the numerical missing-data rule.

In [ ]:

print(df_clean["Age"].describe())
print("Invalid age records:")
print(df_clean.loc[df_clean["Age"] < 0, ["Sample_ID", "Age"]])

# Do not invent a specific corrected age.
df_clean.loc[df_clean["Age"] < 0, "Age"] = np.nan


## Task 6 — Outlier Detection

The IQR rule defines lower and upper boundaries as Q1 − 1.5×IQR and Q3 + 1.5×IQR. GeneA in S009 is the only potential outlier. It is retained but flagged because an outlier can represent a true biological signal, a technical artifact, or a data-entry error.

In [ ]:

for gene in ["GeneA", "GeneB", "GeneC"]:
    print(f"\n{gene}")
    print(df_clean[gene].describe())

    plt.figure(figsize=(6,4))
    plt.boxplot(df_clean[gene].dropna())
    plt.ylabel(f"{gene} Expression")
    plt.title(f"{gene} Expression Distribution")
    plt.show()


In [ ]:

for gene in ["GeneA", "GeneB", "GeneC"]:
    q1 = df_clean[gene].quantile(0.25)
    q3 = df_clean[gene].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = df_clean[(df_clean[gene] < lower) | (df_clean[gene] > upper)]
    print(f"{gene}: Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}, "
          f"lower={lower:.2f}, upper={upper:.2f}")
    print(outliers[["Sample_ID", gene]].to_string(index=False) if not outliers.empty else "No outliers")


## Task 7 — Data Transformation

The mean expression summarizes the three genes for each sample. Log2(x+1) reduces the influence of very large values and is commonly used for skewed abundance/expression measurements.

In [ ]:

# Impute numerical missing values using medians.
for col in ["Age", "GeneB", "GeneC"]:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

df_clean["MeanExpression"] = df_clean[["GeneA", "GeneB", "GeneC"]].mean(axis=1)

for gene in ["GeneA", "GeneB", "GeneC"]:
    df_clean[f"{gene}_log2"] = np.log2(df_clean[gene] + 1)

print(df_clean[["Sample_ID","GeneA","GeneA_log2","GeneB","GeneB_log2","GeneC","GeneC_log2","MeanExpression"]])


## Task 8 — Reshape the Dataset

Wide expression data are converted to long format using `melt()`. Long format is useful for plotting and statistical models because gene identity becomes a variable rather than a separate column.

In [ ]:

long_df = df_clean.melt(
    id_vars=["Sample_ID", "Species", "Tissue", "Sex", "Age", "Treatment"],
    value_vars=["GeneA", "GeneB", "GeneC"],
    var_name="Gene",
    value_name="Expression"
)

print("Shape before reshaping:", df_clean.shape)
print("Shape after reshaping:", long_df.shape)
long_df.head(15)


## Task 9 — Final Data Validation

In [ ]:

df_clean.info()
print("\nMissing values:")
print(df_clean.isnull().sum())
print("\nDuplicate records:", df_clean.duplicated().sum())
print("\nSpecies:", df_clean["Species"].unique())
print("Tissue:", df_clean["Tissue"].unique())
print("Sex:", df_clean["Sex"].unique())
print("Treatment:", df_clean["Treatment"].unique())
print("\nDescriptive statistics:")
print(df_clean.describe())


In [ ]:

before_after = pd.DataFrame({
    "Quality Measure": [
        "Number of rows", "Duplicate records", "Missing values",
        "Invalid ages", "Tissue categories", "Sex categories",
        "Treatment categories", "Potential outliers"
    ],
    "Before Cleaning": [
        len(df), df.duplicated().sum(), int(df.isnull().sum().sum()),
        int((df["Age"] < 0).sum()), df["Tissue"].nunique(),
        df["Sex"].nunique(), df["Treatment"].nunique(), 1
    ],
    "After Cleaning": [
        len(df_clean), df_clean.duplicated().sum(), int(df_clean.isnull().sum().sum()),
        int((df_clean["Age"] < 0).sum()), df_clean["Tissue"].nunique(),
        df_clean["Sex"].nunique(), df_clean["Treatment"].nunique(), 1
    ]
})
before_after



## Raw Sequencing Data Collection

The assignment also requires finding a published paper, locating its raw SRA data, downloading the raw data, and converting it to FASTQ. This part is kept separate from the supplied 16-row messy dataset because no paper/SRA accession is specified in the assignment source. A reproducible example can be added after selecting a published study and accession.

Typical commands are:

```bash
prefetch SRR_ACCESSION
fasterq-dump SRR_ACCESSION --split-files --outdir fastq/
```

The accession and paper should be documented in the final report.


In [ ]:

# Save final deliverables
df_clean.to_csv("cleaned_bioinformatics_data.csv", index=False)
long_df.to_csv("long_format_expression.csv", index=False)
before_after.to_csv("before_after_quality_summary.csv", index=False)

print("Saved:")
print("- cleaned_bioinformatics_data.csv")
print("- long_format_expression.csv")
print("- before_after_quality_summary.csv")



## Conclusions

The dataset contained missing values, an exact duplicate, inconsistent categorical labels, an impossible negative age, and one extreme GeneA observation. The duplicate was removed; categorical labels were standardized; missing numerical values were median-imputed after inspection; the impossible age was treated as missing before imputation; and the GeneA outlier was retained and flagged rather than automatically deleted. The final expression data were also transformed to long format.
